In [ ]:
import torch
from IPython.display import HTML, display

if not torch.cuda.is_available():
    raise SystemExit("⚠️ GPU Not Detected! Enable GPU in Settings → Accelerator.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
cuda_ver = torch.version.cuda

display(HTML(f"""
<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
  <h3 style='color:#2e7d32;margin:0 0 8px 0;'>✅ Hardware Accelerator Ready</h3>
  <table style='font-family:monospace;font-size:13px;'>
    <tr><td><b>GPU Device:</b></td><td>&nbsp;{gpu_name}</td></tr>
    <tr><td><b>Total VRAM:</b></td><td>&nbsp;{vram_gb} GB</td></tr>
    <tr><td><b>CUDA Version:</b></td><td>&nbsp;{cuda_ver}</td></tr>
    <tr><td><b>PyTorch:</b></td><td>&nbsp;{torch.__version__}</td></tr>
  </table>
</div>
"""))

In [ ]:
import os
from IPython.display import HTML, display

print("🚀 Setting up ComfyUI & YuE2 environment...")
os.chdir('/kaggle/working')

if not os.path.exists("/kaggle/working/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI

%cd /kaggle/working/ComfyUI

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio accelerate sentencepiece mutagen websocket-client safetensors ipywidgets librosa
!apt-get -y install -qq aria2 ffmpeg > /dev/null 2>&1

display(HTML("<div style='color:green;'><b>✅ Environment installed!</b></div>"))

In [ ]:
import os
import time
import subprocess
import urllib.request
from IPython.display import HTML, display

%cd /kaggle/working/ComfyUI

model_dir = "/kaggle/working/ComfyUI/models/checkpoints"
encoder_dir = "/kaggle/working/ComfyUI/models/audio_encoders"
os.makedirs(model_dir, exist_ok=True)
os.makedirs(encoder_dir, exist_ok=True)

model_path = os.path.join(model_dir, "yue2_3b_int8_convrot.safetensors")
encoder_path = os.path.join(encoder_dir, "sheetsage2_bf16.safetensors")

if not os.path.exists(model_path) or os.path.getsize(model_path) < 4_000_000_000:
    print("📥 Downloading YuE2-3B INT8 (~4.5 GB)...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Comfy-Org/YuE2/resolve/main/checkpoints/yue2_3b_int8_convrot.safetensors" \
      -d "{model_dir}" -o "yue2_3b_int8_convrot.safetensors"
else:
    print("⚡ INT8 checkpoint already in cache.")

if not os.path.exists(encoder_path) or os.path.getsize(encoder_path) < 1_000_000_000:
    print("📥 Downloading SheetSage2 Encoder (~1.8 GB)...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Comfy-Org/YuE2/resolve/main/audio_encoders/sheetsage2_bf16.safetensors" \
      -d "{encoder_dir}" -o "sheetsage2_bf16.safetensors"
else:
    print("⚡ SheetSage2 encoder already in cache.")

!pkill -9 -f "main.py" || True
time.sleep(2)

server_log = open("/kaggle/working/comfy_server.log", "w")
server_cmd = ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--highvram", "--dont-upcast-attention"]
proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=server_log, cwd="/kaggle/working/ComfyUI")

server_alive = False
for i in range(40):
    if proc.poll() is not None: break
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        server_alive = True
        break
    except Exception:
        time.sleep(1.5)

if server_alive:
    display(HTML("<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'><b style='color:#2e7d32;'>🟢 YuE2-3B INT8 Engine Online!</b><br><span style='color:#555;'>Run Block 4.</span></div>"))
else:
    print("❌ Startup failed. Log (/kaggle/working/comfy_server.log):")
    with open("/kaggle/working/comfy_server.log", "r") as f:
        print(f.read())

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import json, time, random, glob, os, uuid, subprocess, websocket, urllib.request, base64

ckpt_name = "yue2_3b_int8_convrot.safetensors"

preset_dict = {
    "Indie Pop Anthem": "Upbeat indie pop with warm female vocals, bright electric guitars, punchy drums, melodic bass, and subtle synth layers. Catchy and energetic, with an uplifting summer atmosphere, a memorable chorus, and polished modern production.",
    "Synthwave 80s Retro": "1980s retro synthwave, analog vintage synthesizers, gated snare reverb, driving arpeggiated bassline, male nostalgic vocals, neon aesthetic, cinematic nocturnal drive.",
    "Lo-Fi Chill Hop": "Warm lofi hip hop, vinyl crackle, mellow electric piano chords, smooth relaxed boom bap drums, deep warm sub bass, peaceful afternoon coffee shop atmosphere.",
    "Cinematic Epic Orchestral": "Epic cinematic trailer music, soaring brass ensemble, thundering Taiko drums, expressive emotional strings, dramatic crescendos, heroic orchestral climax.",
    "Modern Alt Rock": "Modern alternative rock, overdriven crunchy guitars, aggressive live acoustic drum kit, energetic lead vocals, heavy bassline, anthemic chorus."
}

# ---------- Виджеты ----------
mode_selector = widgets.Dropdown(
    options=["🎵 Generate New Song", "🔄 Create Cover"],
    value="🎵 Generate New Song",
    description='Mode:', style={'description_width': 'initial'}
)

genre_dd = widgets.Dropdown(
    options=["Custom"] + list(preset_dict.keys()),
    value="Indie Pop Anthem",
    description='Genre:', style={'description_width': 'initial'}
)
style_ta = widgets.Textarea(
    value=preset_dict["Indie Pop Anthem"],
    description='Style:',
    layout=widgets.Layout(width='100%', height='80px'),
    style={'description_width': 'initial'}
)
lyrics_ta = widgets.Textarea(
    value="[Verse]\nMorning light across the window\nCity waking down below\nI can hear the streets are calling\nFeels like somewhere we should go\n\n[Chorus]\nRun with me into the sunlight\nLeave the shadows far behind\nWe don't need to know tomorrow\nTonight the whole world feels alive\n\n[Outro]",
    description='Lyrics:',
    layout=widgets.Layout(width='100%', height='150px'),
    style={'description_width': 'initial'}
)
group_new = widgets.VBox([genre_dd, style_ta, lyrics_ta])

audio_up = widgets.FileUpload(accept='.wav,.mp3', multiple=False, description='Source Audio:')
cov_style = widgets.Text(
    value="Jazz-funk cover, smooth saxophone, upright bass",
    description='Target Style:', style={'description_width': 'initial'}
)
cov_lyrics = widgets.Textarea(
    value="",
    description='Lyrics (Optional):',
    layout=widgets.Layout(width='100%', height='100px'),
    placeholder="Leave empty to keep original structure"
)
group_cover = widgets.VBox([audio_up, cov_style, cov_lyrics], visible=False)

abc_cb = widgets.Checkbox(value=False, description='Enable ABC Planning (slower, better structure)')
samp_dd = widgets.Dropdown(options=["dpm_2","euler","dpmpp_2m","heun","dpmpp_sde"], value="dpm_2", description='Sampler:')
sched_dd = widgets.Dropdown(options=["sgm_uniform","normal","karras","simple"], value="sgm_uniform", description='Scheduler:')
steps_sl = widgets.IntSlider(value=32, min=10, max=50, step=1, description='Steps:')
fmt_rb = widgets.RadioButtons(options=['mp3','wav','flac'], value='mp3', description='Format:')
settings_acc = widgets.Accordion(
    children=[widgets.VBox([abc_cb, samp_dd, sched_dd, steps_sl, fmt_rb])],
    titles=['⚙️ Advanced Settings']
)

gen_btn = widgets.Button(description='🚀 Start Generation', button_style='primary',
                         layout=widgets.Layout(width='100%', height='50px'))
out_area = widgets.Output()

# ---------- Обработчики UI ----------
def on_mode_change(change):
    group_new.visible = (change['new'] == "🎵 Generate New Song")
    group_cover.visible = (change['new'] == "🔄 Create Cover")
mode_selector.observe(on_mode_change, names='value')

def update_style(change):
    if change['new'] != "Custom":
        style_ta.value = preset_dict.get(change['new'], "")
genre_dd.observe(update_style, names='value')

# ---------- Устойчивый парсер загруженного аудио ----------
def get_temp_audio(upload_widget):
    v = upload_widget.value
    if not v:
        return None
    name, data = "upload", None
    if isinstance(v, dict):
        if not v:
            return None
        key = next(iter(v))
        item = v[key]
        if isinstance(item, dict):
            data = item.get("content") or item.get("value") or item.get("data")
            name = item.get("name", key)
        else:
            data = item
            name = key
    elif isinstance(v, (list, tuple)):
        if not v:
            return None
        item = v[0]
        if isinstance(item, dict):
            data = item.get("content") or item.get("value") or item.get("data")
            name = item.get("name", "upload")
        else:
            data = item
    else:
        data = v
    if data is None:
        return None
    if isinstance(data, str):
        try:
            data = base64.b64decode(data)
        except Exception:
            return None
    if not isinstance(data, (bytes, bytearray)):
        return None
    safe = "".join(c for c in name if c.isalnum() or c in "._-") or "source.wav"
    if "." not in safe:
        safe += ".wav"
    temp_path = "/tmp/" + str(int(time.time())) + "_" + safe
    with open(temp_path, "wb") as f:
        f.write(bytes(data))
    return temp_path

# ---------- Генерация ----------
def on_generate(b):
    with out_area:
        out_area.clear_output(wait=True)
        is_cover = (mode_selector.value == "🔄 Create Cover")

        try:
            if is_cover:
                src_audio = get_temp_audio(audio_up)
                if not src_audio:
                    raise ValueError("No audio detected. Pick a .wav/.mp3 in 'Source Audio', wait until the filename appears under the button, THEN press Generate.")
                active_style = cov_style.value
                active_lyrics = cov_lyrics.value if cov_lyrics.value.strip() else "[Instrumental]"
                comp_mode = 'melody'
                enable_abc = True
                print("🎧 Cover mode. Source:", os.path.basename(src_audio))
            else:
                lyr = lyrics_ta.value
                if not lyr.strip():
                    raise ValueError("Lyrics required!")
                g_preset = genre_dd.value
                active_style = style_ta.value if g_preset == "Custom" else preset_dict.get(g_preset, style_ta.value)
                active_lyrics = lyr
                comp_mode = 'full'
                enable_abc = abc_cb.value

            prefix = "audio/YuE2_" + str(int(time.time()))
            active_seed = random.randint(1, 2147483647)

            workflow_api = {
              "10": {"inputs": {"filename_prefix": prefix, "format": fmt_rb.value, "format.quality": "320k", "audio": ["33:9", 0]}, "class_type": "SaveAudioAdvanced"},
              "33:15": {"inputs": {"ckpt_name": ckpt_name}, "class_type": "CheckpointLoaderSimple"},
              "33:24": {"inputs": {"style": ["33:36", 0], "lyrics": ["33:37", 0], "seed": ["33:34", 0], "mode": comp_mode, "max_abc_tokens": 8192, "temperature": 0.7, "top_p": 0.9, "top_k": 30, "repetition_penalty": 1.005, "penalty_window": 100, "clip": ["33:15", 1]}, "class_type": "YuE2GenerateABC"},
              "33:14": {"inputs": {"source": ["33:30", 0]}, "class_type": "PreviewAny"},
              "33:30": {"inputs": {"switch": ["33:31", 0], "on_false": ["33:32", 0], "on_true": ["33:24", 0]}, "class_type": "ComfySwitchNode"},
              "33:32": {"inputs": {"value": ""}, "class_type": "PrimitiveString"},
              "33:25": {"inputs": {"style": ["33:36", 0], "lyrics": ["33:37", 0], "abc": ["33:14", 0], "seed": ["33:34", 0], "mode": comp_mode, "max_duration": 360, "temperature": 1.0, "top_p": 0.95, "top_k": 100, "repetition_penalty": 1.2, "clip": ["33:15", 1]}, "class_type": "YuE2GenerateMusic"},
              "33:31": {"inputs": {"value": enable_abc}, "class_type": "PrimitiveBoolean"},
              "33:18": {"inputs": {"conditioning": ["33:25", 0]}, "class_type": "ConditioningZeroOut"},
              "33:23": {"inputs": {"source": ["33:25", 1]}, "class_type": "PreviewAny"},
              "33:8": {"inputs": {"seed": ["33:34", 0], "steps": steps_sl.value, "cfg": 1.0, "sampler_name": samp_dd.value, "scheduler": sched_dd.value, "denoise": 1.0, "model": ["33:15", 0], "positive": ["33:25", 0], "negative": ["33:18", 0], "latent_image": ["33:5", 0]}, "class_type": "KSampler"},
              "33:5": {"inputs": {"seconds": ["33:25", 1], "batch_size": 1}, "class_type": "EmptyYuE2LatentAudio"},
              "33:9": {"inputs": {"samples": ["33:8", 0], "vae": ["33:15", 2]}, "class_type": "VAEDecodeAudio"},
              "33:34": {"inputs": {"seed": active_seed}, "class_type": "SeedNode"},
              "33:36": {"inputs": {"value": active_style}, "class_type": "PrimitiveStringMultiline"},
              "33:37": {"inputs": {"value": active_lyrics}, "class_type": "PrimitiveStringMultiline"}
            }

            node_names = {
                "33:15": "Loading checkpoint",
                "33:24": "ABC symbolic planning",
                "33:30": "ABC route switch",
                "33:25": "Music synthesis",
                "33:8": "KSampler diffusion",
                "33:9": "VAE decode to audio",
                "10": "Saving file"
            }

            client_id = str(uuid.uuid4())
            ws = websocket.WebSocket()
            ws.connect("ws://127.0.0.1:8188/ws?clientId=" + client_id)
            req_payload = json.dumps({"prompt": workflow_api, "client_id": client_id}).encode('utf-8')
            req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_payload, headers={'Content-Type': 'application/json'})
            resp = urllib.request.urlopen(req)
            prompt_id = json.loads(resp.read())['prompt_id']

            print("📡 Job:", prompt_id[:8], "... Seed:", active_seed)
            start_time = time.time()

            while True:
                msg = ws.recv()
                if isinstance(msg, str):
                    event = json.loads(msg)
                    evt_type = event.get('type')
                    evt_data = event.get('data', {})

                    if evt_type == 'execution_error':
                        raise Exception(evt_data.get('exception_message', 'Unknown error'))

                    if evt_type == 'executing':
                        node_id = evt_data.get('node')
                        if node_id is None:
                            if evt_data.get('prompt_id') == prompt_id:
                                break
                        else:
                            print("▶️ Stage:", node_names.get(str(node_id), "Node " + str(node_id)))

                    elif evt_type == 'progress':
                        curr = evt_data.get('value', 0)
                        mx = evt_data.get('max', steps_sl.value)
                        if mx > 0:
                            pct = int((curr / mx) * 100)
                            filled = int(25 * curr // mx)
                            bar = '█' * filled + '░' * (25 - filled)
                            print("\r   [" + bar + "] " + str(curr) + "/" + str(mx) + " (" + str(pct) + "%)", end="", flush=True)

            ws.close()
            elapsed = round(time.time() - start_time, 1)
            print("\n⏱️ Total: " + str(elapsed) + "s")

            files = glob.glob("/kaggle/working/ComfyUI/output/" + prefix + "*." + fmt_rb.value)
            if files:
                latest = max(files, key=os.path.getctime)
                size_mb = round(os.path.getsize(latest) / (1024 * 1024), 2)
                with open(latest, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode()
                mime = "audio/mpeg" if fmt_rb.value == "mp3" else "audio/" + fmt_rb.value
                kind = "Cover" if is_cover else "New Song"
                display(HTML(
                    "<div style='background:#121214;color:white;padding:15px;border-radius:8px;margin-top:10px;'>"
                    "<h3 style='margin:0 0 10px 0;'>✅ " + kind + " Ready (" + str(elapsed) + "s, " + str(size_mb) + " MB)</h3>"
                    "<audio controls autoplay src='data:" + mime + ";base64," + b64 + "' style='width:100%;'></audio>"
                    "<p style='margin:8px 0 0 0;'><small>File: " + os.path.basename(latest) + "</small></p>"
                    "</div>"
                ))
            else:
                print("⚠️ No output file found.")

        except Exception as e:
            print("❌ Error:", e)
            import traceback
            traceback.print_exc()

gen_btn.on_click(on_generate)

display(mode_selector)
display(group_new)
display(group_cover)
display(settings_acc)
display(gen_btn)
display(out_area)

In [ ]:
import glob
import os
import base64
import zipfile
import time
from IPython.display import display, Javascript, HTML, FileLink

def smart_download_tracks():
    search_paths = [
        "/kaggle/working/ComfyUI/output/**/*.mp3",
        "/kaggle/working/ComfyUI/output/**/*.wav",
        "/kaggle/working/ComfyUI/output/**/*.flac",
        "/kaggle/working/ComfyUI/output/*.mp3",
        "/kaggle/working/ComfyUI/output/*.wav",
        "/kaggle/working/ComfyUI/output/*.flac",
        "/kaggle/working/*.mp3",
        "/kaggle/working/*.wav",
        "/kaggle/working/*.flac",
    ]
    all_audio = []
    for p in search_paths:
        all_audio.extend(glob.glob(p, recursive=True))
    all_audio = sorted(set(all_audio), key=os.path.getmtime, reverse=True)

    if not all_audio:
        display(HTML(
            "<div style='background:#fff3cd;padding:15px;border-radius:8px;"
            "border-left:5px solid #ffc107;'><b>⚠️ No audio files found.</b> "
            "Run Block 4 first.</div>"
        ))
        return

    print("📦 Found {} track(s):".format(len(all_audio)))
    total = 0.0
    for t in all_audio:
        mb = round(os.path.getsize(t) / (1024 * 1024), 2)
        total += mb
        ct = time.strftime('%H:%M:%S', time.localtime(os.path.getmtime(t)))
        print("  • {:<48} {:>6.2f} MB  [{}]".format(os.path.basename(t), mb, ct))
    print("  " + "-" * 45)
    print("  TOTAL: {:.2f} MB".format(total))
    print()

    zip_path = "/kaggle/working/YuE2_All_Tracks.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)

    print("📦 Packing archive...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in all_audio:
            zf.write(f, arcname=os.path.basename(f))

    size_mb = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
    print("✅ Archive ready: {:.2f} MB at {}".format(size_mb, zip_path))
    print()

    SAFE_LIMIT_MB = 80
    auto_triggered = False

    if size_mb <= SAFE_LIMIT_MB:
        print("🚀 Triggering auto-download via JS Blob...")
        try:
            with open(zip_path, "rb") as f:
                raw = f.read()
            b64 = base64.b64encode(raw).decode("ascii")

            js = """
            (function() {
                try {
                    var b64 = "%s";
                    var bin = atob(b64);
                    var len = bin.length;
                    var bytes = new Uint8Array(len);
                    for (var i = 0; i < len; i++) bytes[i] = bin.charCodeAt(i);
                    var blob = new Blob([bytes], {type: 'application/zip'});
                    var url = URL.createObjectURL(blob);
                    var a = document.createElement('a');
                    a.href = url;
                    a.download = 'YuE2_All_Tracks.zip';
                    document.body.appendChild(a);
                    a.click();
                    setTimeout(function() {
                        document.body.removeChild(a);
                        URL.revokeObjectURL(url);
                    }, 200);
                    console.log('[YuE2] Auto-download triggered.');
                } catch(e) {
                    console.error('[YuE2] Blob failed:', e);
                }
            })();
            """ % b64

            display(Javascript(js))
            auto_triggered = True
            print("✅ Auto-download sent to browser. Check Downloads tab.")
        except Exception as e:
            print("⚠️ Blob method error: {}".format(e))
    else:
        print("ℹ️ Archive {:.2f} MB > {} MB limit. Skipping auto-download.".format(size_mb, SAFE_LIMIT_MB))

    print()
    if auto_triggered:
        note = "If the auto-download didn't start (popup blocker?), use the link or panel below."
    else:
        note = "Use the link or the Output panel below to download manually."

    display(HTML(
        "<div style='background:#e8f5e9;padding:16px;border-radius:10px;"
        "border-left:6px solid #4caf50;margin:12px 0;font-family:sans-serif;'>"
        "<h3 style='margin:0 0 10px 0;color:#2e7d32;'>⬇️ Download Options</h3>"
        "<p style='margin:4px 0;font-size:14px;'>" + note + "</p>"
        "<p style='margin:6px 0;'><b>Option A:</b> click the blue link below.</p>"
        "<p style='margin:6px 0;'><b>Option B (100% works):</b> right sidebar → "
        "<b>Output</b> tab → <code>YuE2_All_Tracks.zip</code> → ⬇️ arrow.</p>"
        "<p style='margin:6px 0;font-size:12px;color:#666;'>"
        "Archive: <b>YuE2_All_Tracks.zip</b> • {:.2f} MB • {} tracks</p>"
        "</div>".format(size_mb, len(all_audio))
    ))

    display(FileLink(zip_path))

    if len(all_audio) <= 20:
        display(HTML(
            "<div style='background:#f5f5f5;padding:10px;border-radius:8px;"
            "margin:10px 0;'><b>🎵 Individual track links:</b></div>"
        ))
        for t in all_audio:
            mb = round(os.path.getsize(t) / (1024 * 1024), 2)
            print("  • {} ({:.2f} MB)".format(os.path.basename(t), mb))
            display(FileLink(t))

smart_download_tracks()